# Phase 9  -  Gradient Boosting (XGBoost + LightGBM)

**Research questions addressed:**
- **Q1** (multi-class): Can defence system repertoire classify species?
  How does gradient boosting compare to Random Forest?
- **Q2** (binary, per species): Can defence profile predict high-ARG burden?

**Methodology highlights:**
- All evaluation uses `StratifiedGroupKFold` (5-fold, groups = Mash phylogroups from Phase 6)
- Feature set: identical 265 dp_* features used in Phases 7–8 (specificity filter applied)
- Hyperparameter tuning: GridSearchCV (fixed n_estimators) to find best tree structure,
  then early stopping to set n_estimators in the final model
- Statistical comparison vs RF: McNemar's test on pooled per-genome predictions (n=878)
- Calibration: reliability diagram + Brier score for XGBoost and LightGBM

**Learning thread:**
Concepts introduced: sequential boosting vs parallel bagging, gradient descent in function
space, early stopping and the CV-leakage trap, probability calibration, statistical model
comparison. Each section includes a grounded question before the code runs.


## Section 1  -  Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    confusion_matrix, roc_auc_score, brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.dummy import DummyClassifier
import joblib

from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

ROOT  = Path("..")
PROC  = ROOT / "data" / "processed"
RES   = ROOT / "results"
FIG   = RES / "figures" / "gb"
FIG.mkdir(parents=True, exist_ok=True)
(RES / "models").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
N_BOOT       = 2000

print(f"XGBoost version: {__import__('xgboost').__version__}")
print(f"LightGBM version: {lgb.__version__}")
print("Imports OK.")


XGBoost version: 3.2.0
LightGBM version: 4.6.0
Imports OK.


## Section 2  -  Load data and feature selection

Identical 265-feature set from Phases 7–8. Using the same features across all models
is non-negotiable for valid performance comparison.


In [2]:
fm      = pd.read_parquet(PROC / "feature_matrix_3460.parquet")
dp_cols    = sorted([c for c in fm.columns if c.startswith("dp_")])
sp_prev    = fm.groupby("species")[dp_cols].mean()
spec_score = sp_prev.std() / 0.5
markers    = spec_score[spec_score >= 0.70].index.tolist()
FEAT_COLS  = [c for c in dp_cols if c not in markers]

y_q1_str = fm["species"].to_numpy(dtype=str)
groups   = fm["phylogroup"].to_numpy(dtype=str)
X        = fm[FEAT_COLS].to_numpy(dtype=float)

# Integer-encode labels: XGBoost requires integer class labels (or sklearn auto-handles it,
# but explicit encoding avoids edge cases across XGBoost versions)
le = LabelEncoder()
y_q1 = le.fit_transform(y_q1_str)   # 0..5
CLASSES = list(le.classes_)         # alphabetical: abaumannii, ecloaceae, ...

print(f"Feature matrix: {X.shape[0]} genomes x {X.shape[1]} features")
print(f"Classes ({len(CLASSES)}): {CLASSES}")
print(f"Labels encoded: {dict(zip(CLASSES, le.transform(CLASSES)))}")
print(f"Markers removed: {len(markers)}")


Feature matrix: 878 genomes x 265 features
Classes (6): [np.str_('abaumannii'), np.str_('ecloaceae'), np.str_('efaecium'), np.str_('kpneumoniae'), np.str_('paeruginosa'), np.str_('saureus')]
Labels encoded: {np.str_('abaumannii'): np.int64(0), np.str_('ecloaceae'): np.int64(1), np.str_('efaecium'): np.int64(2), np.str_('kpneumoniae'): np.int64(3), np.str_('paeruginosa'): np.int64(4), np.str_('saureus'): np.int64(5)}
Markers removed: 9


## Section 3  -  GroupedStratifiedKFold (same as Phases 7–8)

No new concepts here  -  this is the same splitter established in Phase 6.
The key constraint: all genomes from one phylogroup land in the same fold.


In [3]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

import sys as _sys
_sys.path.insert(0, str(Path("..") / "src"))
from evaluation.bootstrap import bootstrap_ci_auto as _bootstrap_ci_auto

def bootstrap_ci(y_true, y_pred, groups_arg=None, metric_fn=None, n_boot=N_BOOT, seed=42):
    """
    Cluster bootstrap CI where n_phylogroups >= 15 (Q1, Q2-EC/KP/PA).
    Falls back to genome-level bootstrap when n_phylogroups < 15 (EF, SA, AB).
    Wrapper maintains the original (mean, lo, hi) return convention.
    """
    if metric_fn is None:
        metric_fn = balanced_accuracy_score
    mean_val = metric_fn(y_true, y_pred)
    if groups_arg is None:
        # No group info: genome-level (legacy path, should not be used for new calls)
        import numpy as _np
        rng = _np.random.RandomState(seed)
        n = len(y_true)
        scores = [metric_fn(y_true[rng.randint(0, n, n)], y_pred[rng.randint(0, n, n)])
                  for _ in range(n_boot)]
        lo, hi = _np.percentile(scores, [2.5, 97.5])
    else:
        lo, hi, _ = _bootstrap_ci_auto(y_true, y_pred, groups_arg, metric_fn, n_boot, seed)
    return mean_val, lo, hi


# Quick fold-size check
fold_sizes = [len(te) for _, te in cv.split(X, y_q1, groups=groups)]
print(f"Fold sizes: {fold_sizes}  (range {min(fold_sizes)}-{max(fold_sizes)})")


Fold sizes: [232, 212, 178, 133, 123]  (range 123-232)


## Section 5  -  XGBoost quickrun (no early stopping, default-like params)

Before tuning, run XGBoost with reasonable defaults and no early stopping.
This gives a reference point: does gradient boosting outperform RF=0.878 at all?

**Parameters:**
- `n_estimators=200`: 200 trees (we'll improve this with early stopping later)
- `learning_rate=0.1`: standard starting point
- `max_depth=6`: moderately deep (RF used max_depth=20; shallower is better for boosting)
- `subsample=0.8`: subsample 80% of genomes per tree (reduces overfitting, adds randomness)
- `colsample_bytree=0.8`: subsample 80% of features per tree

**No class_weight here:** XGBoost handles class imbalance via `sample_weight` in fit(),
not via a constructor parameter. Species distribution is ~150/species so imbalance is
mild; compute sample weights as inverse class frequency.


In [4]:
from sklearn.utils.class_weight import compute_sample_weight

xgb_quick = XGBClassifier(
    n_estimators     = 200,
    learning_rate    = 0.1,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = "mlogloss",
    verbosity        = 0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
)

ba_scores = []
yt_quick = np.empty(len(y_q1), dtype=int)
yp_quick = np.empty(len(y_q1), dtype=int)
for tr, te in cv.split(X, y_q1, groups=groups):
    sw = compute_sample_weight("balanced", y_q1[tr])
    xgb_quick.fit(X[tr], y_q1[tr], sample_weight=sw)
    pred = xgb_quick.predict(X[te])
    yt_quick[te] = y_q1[te]
    yp_quick[te] = pred
    ba_scores.append(balanced_accuracy_score(y_q1[te], pred))

mean_ba, lo, hi = bootstrap_ci(yt_quick, yp_quick)
print(f"XGBoost quickrun (grouped CV): BA={mean_ba:.4f} [{lo:.4f}-{hi:.4f}]")
print(f"RF Phase 8 reference:          BA=0.8780 [0.8590-0.8980]")
print(f"Delta XGB_quick vs RF:         {mean_ba - 0.8780:+.4f}")


XGBoost quickrun (grouped CV): BA=0.7906 [0.7665-0.8149]
RF Phase 8 reference:          BA=0.8780 [0.8590-0.8980]
Delta XGB_quick vs RF:         -0.0874


## Section 6  -  Early stopping and the CV trap

Early stopping trains up to `n_estimators=500` but halts when validation loss does not improve for `early_stopping_rounds` consecutive trees. Effective `n_estimators = best_iteration_` after fitting.

**CV trap:** Using the test fold as the early stopping validation set leaks test data into the stopping criterion, producing optimistically biased accuracy. Correct approach: hold out a random 20% of the training fold as the early stopping set; test fold remains unseen.

In [5]:
# Demonstrate early stopping with correct inner-fold split
# Use fold 0 for the demonstration
tr_idx, te_idx = next(cv.split(X, y_q1, groups=groups))

# Split training fold: 80% train, 20% val for early stopping
inner_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_inner_rel, val_inner_rel = next(inner_split.split(X[tr_idx], y_q1[tr_idx]))
tr_inner  = tr_idx[tr_inner_rel]
val_inner = tr_idx[val_inner_rel]

sw_inner = compute_sample_weight("balanced", y_q1[tr_inner])

xgb_es_demo = XGBClassifier(
    n_estimators         = 500,
    early_stopping_rounds = 30,
    learning_rate        = 0.1,
    max_depth            = 6,
    subsample            = 0.8,
    colsample_bytree     = 0.8,
    eval_metric          = "mlogloss",
    verbosity            = 0,
    random_state         = RANDOM_STATE,
    n_jobs               = -1,
)

xgb_es_demo.fit(
    X[tr_inner], y_q1[tr_inner],
    sample_weight = sw_inner,
    eval_set      = [(X[val_inner], y_q1[val_inner])],
    verbose       = False,
)

print(f"Trees trained (n_estimators cap): 500")
print(f"Best iteration (early stopping):  {xgb_es_demo.best_iteration}")
print(f"Validation mlogloss at best iter: {xgb_es_demo.best_score:.5f}")
print()
ba_fold0 = balanced_accuracy_score(y_q1[te_idx], xgb_es_demo.predict(X[te_idx]))
print(f"Fold 0 BA with early stopping ({xgb_es_demo.best_iteration} trees): {ba_fold0:.4f}")
print()
print("Compare: XGBoost fixed 200 trees in Section 5 vs optimal early-stopped trees above.")
print("If best_iteration << 200: we were adding wasteful trees in Section 5.")
print("If best_iteration > 200:  we were underfitting in Section 5.")


Trees trained (n_estimators cap): 500
Best iteration (early stopping):  79
Validation mlogloss at best iter: 0.30972

Fold 0 BA with early stopping (79 trees): 0.8374

Compare: XGBoost fixed 200 trees in Section 5 vs optimal early-stopped trees above.
If best_iteration << 200: we were adding wasteful trees in Section 5.
If best_iteration > 200:  we were underfitting in Section 5.


## Section 7  -  XGBoost hyperparameter grid search

**Strategy:** Use GridSearchCV with a *fixed* n_estimators=200 (no early stopping during
grid search) to identify the best (learning_rate, max_depth, subsample) combination.
Then, in Section 8, take those best params and apply early stopping to set n_estimators.

**Why separate the two steps:**
1. GridSearchCV + early stopping + grouped CV + sample weights is a complex interaction
   that is error-prone to implement correctly.
2. Conceptually: tree structure (max_depth, learning_rate, subsample) and number of
   trees (n_estimators via early stopping) are independent hyperparameters.

**Grid:** 3 × 2 × 2 = 12 combinations × 5 folds = 60 XGBoost fits. At n=878, this
runs in ~1-2 minutes.

**Grounded question before running:**
Why is a smaller learning_rate (e.g., 0.05) generally paired with a larger n_estimators?


In [6]:
param_grid_xgb = {
    "learning_rate"   : [0.05, 0.1, 0.3],
    "max_depth"       : [4, 6],
    "subsample"       : [0.8, 1.0],
}

xgb_base = XGBClassifier(
    n_estimators     = 200,          # fixed for grid search
    colsample_bytree = 0.8,
    eval_metric      = "mlogloss",
    verbosity        = 0,
    random_state     = RANDOM_STATE,
    n_jobs           = 1,            # 1 inside; GridSearchCV handles outer parallelism
)

grid_xgb = GridSearchCV(
    estimator  = xgb_base,
    param_grid = param_grid_xgb,
    cv         = cv,
    scoring    = "balanced_accuracy",
    n_jobs     = -1,
    verbose    = 1,
    refit      = True,
)

# M3 fix: sample_weight NOT passed to GridSearchCV  -  weights computed once on full data
# encode class imbalance of held-out test genomes, a minor form of leakage.
# Hyperparameter tuning uses balanced_accuracy scoring which is already imbalance-aware.
# Per-fold sample weights are correctly computed inside the early-stopping CV loop below.
grid_xgb.fit(X, y_q1, groups=groups)

print("\nBest hyperparameters (XGBoost, GridSearchCV):")
for k, v in grid_xgb.best_params_.items():
    print(f"  {k:<22} {v}")
print(f"\nBest CV balanced accuracy: {grid_xgb.best_score_:.4f}")

# Show top 5 combinations
cv_results = pd.DataFrame(grid_xgb.cv_results_)
top5 = (cv_results
        .sort_values("mean_test_score", ascending=False)
        .head(5)[["param_learning_rate", "param_max_depth", "param_subsample",
                  "mean_test_score", "std_test_score"]])
print("\nTop 5 parameter combinations:")
print(top5.to_string(index=False))


Fitting 5 folds for each of 12 candidates, totalling 60 fits



Best hyperparameters (XGBoost, GridSearchCV):
  learning_rate          0.1
  max_depth              4
  subsample              0.8

Best CV balanced accuracy: 0.8523

Top 5 parameter combinations:
 param_learning_rate  param_max_depth  param_subsample  mean_test_score  std_test_score
                0.10                4              0.8         0.852289        0.024531
                0.05                4              1.0         0.849518        0.022619
                0.10                4              1.0         0.849185        0.021767
                0.05                4              0.8         0.848132        0.025557
                0.05                6              0.8         0.845605        0.021409


## Section 8  -  Final XGBoost: best params + early stopping (Q1)

Take the best hyperparameters from Section 7. Now apply early stopping in a proper
grouped CV loop to:
1. Set n_estimators adaptively (no guessing)
2. Get final performance estimate with 95% CI
3. Collect per-genome predictions for McNemar test (Section 10)

**Implementation note  -  early stopping inside CV:**
A new XGBClassifier instance is created at each fold. This is required: if we reuse the
same instance, `early_stopping_rounds` state from a previous fold's `fit()` persists
and corrupts the iteration count. Recreating the instance resets internal state cleanly.


In [7]:
best_params_xgb = grid_xgb.best_params_

ba_scores_xgb = []
f1_scores_xgb = []
best_iters    = []

# For McNemar test: need aligned (genome_idx, true, pred) across all folds
mc_true_xgb  = np.empty(len(y_q1), dtype=int)
mc_pred_xgb  = np.empty(len(y_q1), dtype=int)

for tr, te in cv.split(X, y_q1, groups=groups):
    # Inner validation split for early stopping (from training fold only)
    inner_ss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr_rel, val_rel = next(inner_ss.split(X[tr], y_q1[tr]))
    tr_inner  = tr[tr_rel]
    val_inner = tr[val_rel]

    sw_inner = compute_sample_weight("balanced", y_q1[tr_inner])

    # Fresh instance per fold  -  required for early stopping state isolation
    xgb_fold = XGBClassifier(
        **best_params_xgb,
        n_estimators          = 500,
        early_stopping_rounds = 30,
        colsample_bytree      = 0.8,
        eval_metric           = "mlogloss",
        verbosity             = 0,
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
    )

    xgb_fold.fit(
        X[tr_inner], y_q1[tr_inner],
        sample_weight = sw_inner,
        eval_set      = [(X[val_inner], y_q1[val_inner])],
        verbose       = False,
    )

    pred = xgb_fold.predict(X[te])
    ba_scores_xgb.append(balanced_accuracy_score(y_q1[te], pred))
    f1_scores_xgb.append(f1_score(y_q1[te], pred, average="macro"))
    best_iters.append(xgb_fold.best_iteration)
    mc_true_xgb[te] = y_q1[te]
    mc_pred_xgb[te] = pred

mean_ba_xgb, lo_xgb, hi_xgb = bootstrap_ci(mc_true_xgb, mc_pred_xgb, groups_arg=groups)
np.save(RES / 'q1_xgb_pred_true.npy', mc_true_xgb)
np.save(RES / 'q1_xgb_pred_pred.npy', mc_pred_xgb)
np.save(RES / 'q1_xgb_pred_groups.npy', groups)
mean_f1_xgb, lo_f1_xgb, hi_f1_xgb = bootstrap_ci(mc_true_xgb, mc_pred_xgb,
    groups_arg=groups, metric_fn=lambda yt, yp: f1_score(yt, yp, average="macro"))

print(f"=== XGBoost Q1 (best params + early stopping, grouped CV) ===")
print(f"Balanced accuracy: {mean_ba_xgb:.4f} [{lo_xgb:.4f}-{hi_xgb:.4f}]")
print(f"Macro F1:          {mean_f1_xgb:.4f} [{lo_f1_xgb:.4f}-{hi_f1_xgb:.4f}]")
print(f"Best iterations (per fold): {best_iters}  median={np.median(best_iters):.0f}")
print()
print(f"RF Phase 8 reference: BA=0.8780 [0.8590-0.8980]")
print(f"Delta XGB vs RF:      {mean_ba_xgb - 0.8780:+.4f}")


=== XGBoost Q1 (best params + early stopping, grouped CV) ===
Balanced accuracy: 0.8165 [0.7932-0.8393]
Macro F1:          0.8127 [0.7880-0.8365]
Best iterations (per fold): [124, 143, 236, 178, 132]  median=143

RF Phase 8 reference: BA=0.8780 [0.8590-0.8980]
Delta XGB vs RF:      -0.0615


## Section 9  -  LightGBM

**LightGBM vs XGBoost  -  three implementation differences that matter:**

1. **Leaf-wise vs level-wise growth:**
   XGBoost grows trees level by level (all leaves at a given depth simultaneously).
   LightGBM grows the single leaf with the highest gain at each step  -  it can produce
   deeper, more complex trees for the same number of leaves. `num_leaves` (not `max_depth`)
   is the primary complexity control.

2. **Histogram-based splitting:**
   LightGBM bins continuous features into buckets (histograms), reducing the number of
   split threshold evaluations. On 265 binary features, the advantage is minimal  - 
   binary features already have only one split threshold each.

3. **Early stopping via callbacks (LightGBM 4.x API):**
   Unlike XGBoost (which takes `early_stopping_rounds` as a constructor parameter),
   LightGBM 4.x uses a callbacks list:
   ```python
   callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
   ```
   Functionally identical; the API difference is a library design choice.

**Parameter choices:**
- `num_leaves=63`: roughly equivalent to `max_depth=6` in XGBoost
  (num_leaves ≈ 2^max_depth / 2 for balanced trees; LightGBM grows asymmetrically)
- `learning_rate`, `subsample`: same as XGBoost best params for fair comparison

**Grounded question before running:**
If LightGBM's leaf-wise growth tends to overfit more aggressively than XGBoost's
level-wise growth, which regularisation parameter would you increase to compensate?


In [8]:
ba_scores_lgbm = []
f1_scores_lgbm = []
best_iters_lgbm = []

mc_true_lgbm  = np.empty(len(y_q1), dtype=int)
mc_pred_lgbm  = np.empty(len(y_q1), dtype=int)

lgbm_params = {
    "learning_rate"   : best_params_xgb.get("learning_rate", 0.1),
    "max_depth"       : best_params_xgb.get("max_depth", 6),
    "num_leaves"      : 63,
    "subsample"       : best_params_xgb.get("subsample", 0.8),
    "subsample_freq"  : 1,      # required for subsample to take effect
    "colsample_bytree": 0.8,
    "class_weight"    : "balanced",
    "n_estimators"    : 500,    # ceiling; early stopping will find optimal
    "verbosity"       : -1,
    "random_state"    : RANDOM_STATE,
    "n_jobs"          : -1,
}

for tr, te in cv.split(X, y_q1, groups=groups):
    inner_ss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr_rel, val_rel = next(inner_ss.split(X[tr], y_q1[tr]))
    tr_inner  = tr[tr_rel]
    val_inner = tr[val_rel]

    lgbm_fold = LGBMClassifier(**lgbm_params)

    lgbm_fold.fit(
        X[tr_inner], y_q1[tr_inner],
        eval_set  = [(X[val_inner], y_q1[val_inner])],
        callbacks = [
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )

    pred = lgbm_fold.predict(X[te])
    ba_scores_lgbm.append(balanced_accuracy_score(y_q1[te], pred))
    f1_scores_lgbm.append(f1_score(y_q1[te], pred, average="macro"))
    best_iters_lgbm.append(lgbm_fold.best_iteration_)
    mc_true_lgbm[te] = y_q1[te]
    mc_pred_lgbm[te] = pred

mean_ba_lgbm, lo_lgbm, hi_lgbm = bootstrap_ci(mc_true_lgbm, mc_pred_lgbm, groups_arg=groups)
np.save(RES / 'q1_lgbm_pred_true.npy', mc_true_lgbm)
np.save(RES / 'q1_lgbm_pred_pred.npy', mc_pred_lgbm)
mean_f1_lgbm, lo_f1_lgbm, hi_f1_lgbm = bootstrap_ci(mc_true_lgbm, mc_pred_lgbm,
    groups_arg=groups, metric_fn=lambda yt, yp: f1_score(yt, yp, average="macro"))

print(f"=== LightGBM Q1 (early stopping, grouped CV) ===")
print(f"Balanced accuracy: {mean_ba_lgbm:.4f} [{lo_lgbm:.4f}-{hi_lgbm:.4f}]")
print(f"Macro F1:          {mean_f1_lgbm:.4f} [{lo_f1_lgbm:.4f}-{hi_f1_lgbm:.4f}]")
print(f"Best iterations (per fold): {best_iters_lgbm}  median={np.median(best_iters_lgbm):.0f}")
print()
print("=== Summary so far ===")
lr_ba, lr_lo, lr_hi = 0.8370, 0.8130, 0.8590
rf_ba, rf_lo, rf_hi = 0.8780, 0.8590, 0.8980
print(f"{'Model':<28} {'BA':>6}  {'95% CI':>16}")
print("-" * 52)
print(f"{'LR (Phase 7)':<28} {lr_ba:.4f}  [{lr_lo:.4f}-{lr_hi:.4f}]")
print(f"{'RF (Phase 8)':<28} {rf_ba:.4f}  [{rf_lo:.4f}-{rf_hi:.4f}]")
print(f"{'XGBoost':<28} {mean_ba_xgb:.4f}  [{lo_xgb:.4f}-{hi_xgb:.4f}]")
print(f"{'LightGBM':<28} {mean_ba_lgbm:.4f}  [{lo_lgbm:.4f}-{hi_lgbm:.4f}]")


=== LightGBM Q1 (early stopping, grouped CV) ===
Balanced accuracy: 0.8267 [0.8018-0.8522]
Macro F1:          0.8277 [0.8025-0.8531]
Best iterations (per fold): [59, 61, 100, 92, 99]  median=92

=== Summary so far ===
Model                            BA            95% CI
----------------------------------------------------
LR (Phase 7)                 0.8370  [0.8130-0.8590]
RF (Phase 8)                 0.8780  [0.8590-0.8980]
XGBoost                      0.8165  [0.7932-0.8393]
LightGBM                     0.8267  [0.8018-0.8522]


## Section 10  -  Statistical comparison: McNemar's test

**Why point-estimate comparisons are not enough:**
XGBoost BA=0.882 vs RF BA=0.878 is a difference of 0.004. Is this real or noise?
With 5 fold-level scores, a Wilcoxon test has very low power (n=5). We need more.

**McNemar's test  -  the correct tool:**
McNemar's test operates on paired binary outcomes: for each genome, was model A
correct and model B wrong, or vice versa? The test statistic uses the *discordant*
pairs  -  cases where the models disagree.

Contingency table (n=878 genomes):
```
                  Model B correct   Model B wrong
Model A correct        a                 b
Model A wrong          c                 d
```
McNemar statistic: (b - c)² / (b + c) ~ χ²(1)
Only b and c matter  -  the genomes where the models disagree.
If b ≈ c: the models make errors on the same genomes → no significant difference.
If b >> c (or c >> b): one model is making substantially different errors → real difference.

**What constitutes a meaningful difference here:**
p < 0.05 from McNemar is a statistical threshold. The scientific threshold is whether
the performance gain justifies the added model complexity. For this project:
if delta BA ≤ 0.02 and McNemar p > 0.05, models are equivalent  -  prefer the simpler one.


In [9]:
from scipy.stats import chi2

def mcnemar_test(true, pred_a, pred_b, labels=None):
    """McNemar test comparing two classifiers on the same genomes."""
    correct_a = (true == pred_a)
    correct_b = (true == pred_b)
    b = np.sum(correct_a & ~correct_b)   # A right, B wrong
    c = np.sum(correct_b & ~correct_a)   # B right, A wrong
    if b + c == 0:
        return 1.0, b, c, 0.0
    # Continuity-corrected McNemar
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p    = 1 - chi2.cdf(stat, df=1)
    return p, b, c, stat

# Load RF per-genome predictions: re-run RF CV with same splits to get aligned predictions
# (rf_q1_best.pkl was trained on all data for OOB; we need CV predictions)
rf_best = joblib.load(RES / "models" / "rf_q1_best_3460.pkl")
mc_true_rf = np.empty(len(y_q1), dtype=int)
mc_pred_rf = np.empty(len(y_q1), dtype=int)

for tr, te in cv.split(X, y_q1, groups=groups):
    sw = compute_sample_weight("balanced", y_q1[tr])
    rf_fold_params = {k: v for k, v in rf_best.get_params().items()
                      if k in ["n_estimators", "max_depth", "min_samples_leaf",
                               "max_features", "class_weight", "random_state", "n_jobs"]}
    from sklearn.ensemble import RandomForestClassifier
    rf_fold = RandomForestClassifier(**rf_fold_params)
    rf_fold.fit(X[tr], y_q1[tr])
    mc_true_rf[te] = y_q1[te]
    mc_pred_rf[te] = rf_fold.predict(X[te])

# McNemar pairwise tests
print("=== McNemar's test  -  pairwise model comparison (Q1, n=878) ===")
print()
comparisons = [
    ("XGBoost vs RF",      mc_true_xgb, mc_pred_xgb, mc_true_rf,   mc_pred_rf),
    ("LightGBM vs RF",     mc_true_lgbm, mc_pred_lgbm, mc_true_rf, mc_pred_rf),
    ("XGBoost vs LightGBM", mc_true_xgb, mc_pred_xgb, mc_true_lgbm, mc_pred_lgbm),
]

for label, true_a, pred_a, true_b, pred_b in comparisons:
    # Use model A's true labels (all folds cover same genomes, so any true array is fine)
    p, b, c, stat = mcnemar_test(true_a, pred_a, pred_b)
    acc_a = balanced_accuracy_score(true_a, pred_a)
    acc_b = balanced_accuracy_score(true_b, pred_b)
    sig = "**significant**" if p < 0.05 else "not significant"
    print(f"{label}")
    print(f"  BA: {acc_a:.4f} vs {acc_b:.4f}  delta={acc_a - acc_b:+.4f}")
    print(f"  Discordant pairs: b={b} (A right/B wrong), c={c} (B right/A wrong)")
    print(f"  chi2={stat:.3f}, p={p:.4f}  [{sig}]")
    print()


=== McNemar's test  -  pairwise model comparison (Q1, n=878) ===

XGBoost vs RF
  BA: 0.8165 vs 0.8742  delta=-0.0577
  Discordant pairs: b=21 (A right/B wrong), c=73 (B right/A wrong)
  chi2=27.670, p=0.0000  [**significant**]

LightGBM vs RF
  BA: 0.8267 vs 0.8742  delta=-0.0476
  Discordant pairs: b=23 (A right/B wrong), c=66 (B right/A wrong)
  chi2=19.820, p=0.0000  [**significant**]

XGBoost vs LightGBM
  BA: 0.8165 vs 0.8267  delta=-0.0102
  Discordant pairs: b=35 (A right/B wrong), c=44 (B right/A wrong)
  chi2=0.810, p=0.3681  [not significant]



## Section 10b  -  H3: Fair comparison with fixed n_estimators

**Audit finding H3:** XGBoost and LightGBM use an 80/20 inner split for early
stopping, so they train on only ~64% of the data per fold (80% x 80%).
RF trains on ~80%. The 14pp BA gap (RF=0.878 vs XGB=0.817) may partly reflect
data starvation rather than model quality.

**What this section does:**
Re-run XGB and LGBM with:
- Fixed `n_estimators` from the median of early-stopping best iterations
  (XGB median=143, LGBM median=92)
- No inner validation split -- full training fold used (same effective data as RF)
- Same hyperparameters otherwise

If the gap closes substantially: data starvation was the primary cause.
If the gap persists: RF genuinely outperforms gradient boosting on this dataset.


In [10]:
# H3: fixed n_estimators -- use median from early stopping runs
n_est_xgb  = int(np.median(best_iters))
n_est_lgbm = int(np.median(best_iters_lgbm))
print(f"Fixed n_estimators -- XGB: {n_est_xgb}, LGBM: {n_est_lgbm}")

ba_xgb_fixed  = []
ba_lgbm_fixed = []
mc_pred_xgb_fixed  = np.empty(len(y_q1), dtype=int)
mc_pred_lgbm_fixed = np.empty(len(y_q1), dtype=int)
mc_true_fixed      = np.empty(len(y_q1), dtype=int)

for tr, te in cv.split(X, y_q1, groups=groups):
    sw = compute_sample_weight("balanced", y_q1[tr])
    mc_true_fixed[te] = y_q1[te]

    # XGB: fixed n_estimators, full training fold
    xgb_fixed = XGBClassifier(
        **best_params_xgb,
        n_estimators     = n_est_xgb,
        colsample_bytree = 0.8,
        verbosity        = 0,
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
    )
    xgb_fixed.fit(X[tr], y_q1[tr], sample_weight=sw)
    pred_xf = xgb_fixed.predict(X[te])
    ba_xgb_fixed.append(balanced_accuracy_score(y_q1[te], pred_xf))
    mc_pred_xgb_fixed[te] = pred_xf

    # LGBM: fixed n_estimators, full training fold
    lgbm_fixed = LGBMClassifier(
        **{k: v for k, v in lgbm_params.items() if k != "n_estimators"},
        n_estimators = n_est_lgbm,
    )
    lgbm_fixed.fit(X[tr], y_q1[tr])
    pred_lf = lgbm_fixed.predict(X[te])
    ba_lgbm_fixed.append(balanced_accuracy_score(y_q1[te], pred_lf))
    mc_pred_lgbm_fixed[te] = pred_lf

ba_xf,  lo_xf,  hi_xf  = bootstrap_ci(mc_true_fixed, mc_pred_xgb_fixed, groups_arg=groups)
np.save(RES / 'q1_xgb_fixed_pred_true.npy', mc_true_fixed)
np.save(RES / 'q1_xgb_fixed_pred_pred.npy', mc_pred_xgb_fixed)
np.save(RES / 'q1_xgb_fixed_pred_groups.npy', groups)
ba_lf,  lo_lf,  hi_lf  = bootstrap_ci(mc_true_fixed, mc_pred_lgbm_fixed, groups_arg=groups)

lbl_xf = f"XGB (fixed n={n_est_xgb}, 80% fold)"
lbl_lf = f"LGBM (fixed n={n_est_lgbm}, 80% fold)"

print()
print("=== H3: fixed-n vs early-stopping vs RF (Q1 BA) ===")
print(f"  {'Model':<38} {'BA':>6}  {'95% CI':<18}  {'vs RF delta':>11}")
print("  " + "-" * 78)
ref_rf  = 0.8780
print(f"  {'RF (80% fold, n_estimators free)':<38} {ref_rf:.4f}  [0.8590-0.8980]")
print(f"  {'XGB (early-stop, 64% fold)':<38} {mean_ba_xgb:.4f}  [{lo_xgb:.4f}-{hi_xgb:.4f}]  {mean_ba_xgb - ref_rf:+.4f}")
print(f"  {lbl_xf:<38} {ba_xf:.4f}  [{lo_xf:.4f}-{hi_xf:.4f}]  {ba_xf - ref_rf:+.4f}")
print(f"  {'LGBM (early-stop, 64% fold)':<38} {mean_ba_lgbm:.4f}  [{lo_lgbm:.4f}-{hi_lgbm:.4f}]  {mean_ba_lgbm - ref_rf:+.4f}")
print(f"  {lbl_lf:<38} {ba_lf:.4f}  [{lo_lf:.4f}-{hi_lf:.4f}]  {ba_lf - ref_rf:+.4f}")
print()

# McNemar: fixed-n XGB vs RF
p_xf_rf, b_xf_rf, c_xf_rf, _ = mcnemar_test(mc_true_fixed, mc_pred_xgb_fixed, mc_pred_rf)
sig_xf = "**significant**" if p_xf_rf < 0.05 else "not significant"
print(f"McNemar (XGB fixed-n vs RF): b={b_xf_rf}, c={c_xf_rf}, p={p_xf_rf:.4f}  [{sig_xf}]")
p_lf_rf, b_lf_rf, c_lf_rf, _ = mcnemar_test(mc_true_fixed, mc_pred_lgbm_fixed, mc_pred_rf)
sig_lf = "**significant**" if p_lf_rf < 0.05 else "not significant"
print(f"McNemar (LGBM fixed-n vs RF): b={b_lf_rf}, c={c_lf_rf}, p={p_lf_rf:.4f}  [{sig_lf}]")


Fixed n_estimators -- XGB: 143, LGBM: 92



=== H3: fixed-n vs early-stopping vs RF (Q1 BA) ===
  Model                                      BA  95% CI              vs RF delta
  ------------------------------------------------------------------------------
  RF (80% fold, n_estimators free)       0.8780  [0.8590-0.8980]
  XGB (early-stop, 64% fold)             0.8165  [0.7932-0.8393]  -0.0615
  XGB (fixed n=143, 80% fold)            0.8062  [0.7823-0.8297]  -0.0718
  LGBM (early-stop, 64% fold)            0.8267  [0.8018-0.8522]  -0.0513
  LGBM (fixed n=92, 80% fold)            0.8304  [0.8069-0.8546]  -0.0476

McNemar (XGB fixed-n vs RF): b=21, c=83, p=0.0000  [**significant**]
McNemar (LGBM fixed-n vs RF): b=25, c=65, p=0.0000  [**significant**]


## Section 11  -  Probability calibration

Calibration checks whether model-predicted class probabilities match empirical frequencies. Boosted classifiers typically compress probabilities toward 0.5. For Q2 ARG burden prediction, calibrated probabilities are needed if the output is used for risk scoring. Reliability diagrams for XGBoost and LightGBM are plotted below.

In [11]:
# Use fold 0 for calibration demonstration (same fold as Phase 8 permutation importance)
tr_idx0, te_idx0 = next(cv.split(X, y_q1, groups=groups))

inner_ss0 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_rel0, val_rel0 = next(inner_ss0.split(X[tr_idx0], y_q1[tr_idx0]))
tr_inner0  = tr_idx0[tr_rel0]
val_inner0 = tr_idx0[val_rel0]

# XGBoost fold 0 proba
xgb_cal = XGBClassifier(
    **best_params_xgb,
    n_estimators          = 500,
    early_stopping_rounds = 30,
    colsample_bytree      = 0.8,
    eval_metric           = "mlogloss",
    verbosity             = 0,
    random_state          = RANDOM_STATE,
    n_jobs                = -1,
)
sw0 = compute_sample_weight("balanced", y_q1[tr_inner0])
xgb_cal.fit(X[tr_inner0], y_q1[tr_inner0], sample_weight=sw0,
            eval_set=[(X[val_inner0], y_q1[val_inner0])], verbose=False)
proba_xgb = xgb_cal.predict_proba(X[te_idx0])   # (n_te, 6)

# RF fold 0 proba (for comparison)
from sklearn.ensemble import RandomForestClassifier
rf_cal_params = {k: v for k, v in rf_best.get_params().items()
                 if k in ["n_estimators", "max_depth", "min_samples_leaf",
                          "max_features", "class_weight", "random_state", "n_jobs"]}
rf_cal = RandomForestClassifier(**rf_cal_params)
rf_cal.fit(X[tr_idx0], y_q1[tr_idx0])
proba_rf = rf_cal.predict_proba(X[te_idx0])   # (n_te, 6)

y_te0 = y_q1[te_idx0]

# Plot reliability diagrams: 3 classes (AB worst, SA best, KP middle) for both models
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
classes_to_show = [CLASSES.index(c) for c in ["abaumannii", "kpneumoniae", "saureus"]]

brier_xgb_all, brier_rf_all = [], []

for col, cls_idx in enumerate(classes_to_show):
    y_bin = (y_te0 == cls_idx).astype(int)  # one-vs-rest binary

    for row, (proba, label, color) in enumerate(
        [(proba_xgb, "XGBoost", "#d62728"), (proba_rf, "RF", "#1f77b4")]
    ):
        ax = axes[row][col]
        prob_pos = proba[:, cls_idx]
        frac_pos, mean_pred = calibration_curve(y_bin, prob_pos, n_bins=8, strategy="uniform")
        brier = brier_score_loss(y_bin, prob_pos)
        if row == 0:
            brier_xgb_all.append(brier)
        else:
            brier_rf_all.append(brier)

        ax.plot(mean_pred, frac_pos, "s-", color=color, label=f"{label} (Brier={brier:.3f})")
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect")
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
        ax.set_xlabel("Mean predicted prob")
        ax.set_ylabel("Fraction positive")
        ax.set_title(f"{CLASSES[cls_idx]} (1-vs-rest)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle("Reliability diagrams (fold 0): XGBoost vs RF, selected classes", y=1.02)
plt.tight_layout()
fig.savefig(FIG / "q1_calibration_xgb_vs_rf.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results/figures/gb/q1_calibration_xgb_vs_rf.png")

print(f"\nMean Brier score (3 classes)  -  XGBoost: {np.mean(brier_xgb_all):.4f}  "
      f"RF: {np.mean(brier_rf_all):.4f}")
print("Lower = better calibrated (0 = perfect, ~0.139 = null baseline at 1/6)")


Saved: results/figures/gb/q1_calibration_xgb_vs_rf.png

Mean Brier score (3 classes)  -  XGBoost: 0.0118  RF: 0.0222
Lower = better calibrated (0 = perfect, ~0.139 = null baseline at 1/6)


## Section 12  -  Q2: XGBoost and LightGBM for ARG burden prediction (per species)

**Phase 7–8 Q2 recap:**
LR won 4/6 species; RF won 1 (EF, AUROC 0.900). Conclusion: flexible models overfit
at n~150. The question now: does gradient boosting, with its smoother regularisation via
learning_rate and early stopping, do better than RF on the small-sample species?

**What to look for:**
- EC and KP: LR was dominant (BA 0.752, 0.719). Can XGBoost close the gap?
- EF: RF dominated (BA 0.681). Does boosting match or beat RF here?
- SA and AB: both near chance in Phase 7–8. If boosting improves either, investigate
  leakage  -  resistance in these species is chromosomal, not defence-linked.


In [12]:
results_q2_xgb  = {}
results_q2_lgbm = {}
ba_xgb_dict     = {}  # H4: fold-level XGB BAs per species for BH correction

for sp in sorted(fm["species"].unique()):
    sp_mask  = fm["species"].to_numpy(dtype=str) == sp
    fm_sp    = fm[sp_mask]

    # Q2 labels: use pre-computed arg_burden_tertile (top vs bottom tertile only)
    # Excludes mid_ARG genomes to match the LR baseline task exactly (C1 fix)
    mask_q2 = fm_sp["arg_burden_tertile"].isin(["low_ARG", "high_ARG"])
    fm_q2   = fm_sp[mask_q2]
    if len(fm_q2) < 20 or fm_q2["arg_burden_tertile"].nunique() < 2:
        print(f"  {sp}: insufficient Q2 data -- skip")
        continue
    y_sp   = (fm_q2["arg_burden_tertile"] == "high_ARG").astype(int).values
    # H1: per-species sparsity filter -- keep only features with >=5% prevalence
    feat_prev_sp = fm_q2[FEAT_COLS].mean()
    feat_q2_sp   = feat_prev_sp[feat_prev_sp >= 0.05].index.tolist()
    X_sp         = fm_q2[feat_q2_sp].values
    grp_sp = fm_q2["phylogroup"].to_numpy(dtype=str)
    print(f"  {sp}: {len(feat_q2_sp)} features after H1 filter (from {len(FEAT_COLS)})")

    cv_sp = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for model_name, results_dict in [("xgb", results_q2_xgb), ("lgbm", results_q2_lgbm)]:
        ba_sp, auc_sp = [], []
        all_yt_sp, all_yp_sp = [], []

        for tr, te in cv_sp.split(X_sp, y_sp, groups=grp_sp):
            if len(set(y_sp[te])) < 2:
                continue

            inner_ss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
            tr_rel, val_rel = next(inner_ss.split(X_sp[tr], y_sp[tr]))
            tr_i  = tr[tr_rel]
            val_i = tr[val_rel]
            sw_i  = compute_sample_weight("balanced", y_sp[tr_i])

            if model_name == "xgb":
                m = XGBClassifier(
                    **best_params_xgb,
                    n_estimators=300, early_stopping_rounds=20,
                    colsample_bytree=0.8, eval_metric="logloss",
                    verbosity=0, random_state=RANDOM_STATE, n_jobs=-1,
                )
                m.fit(X_sp[tr_i], y_sp[tr_i], sample_weight=sw_i,
                      eval_set=[(X_sp[val_i], y_sp[val_i])], verbose=False)
            else:
                m = LGBMClassifier(
                    **{k: v for k, v in lgbm_params.items() if k != "n_estimators"},
                    n_estimators=300,
                )
                m.fit(X_sp[tr_i], y_sp[tr_i],
                      eval_set=[(X_sp[val_i], y_sp[val_i])],
                      callbacks=[lgb.early_stopping(20, verbose=False),
                                 lgb.log_evaluation(-1)])

            pred  = m.predict(X_sp[te])
            prob  = m.predict_proba(X_sp[te])[:, 1]
            all_yt_sp.extend(y_sp[te])
            all_yp_sp.extend(pred)
            ba_sp.append(balanced_accuracy_score(y_sp[te], pred))
            auc_sp.append(roc_auc_score(y_sp[te], prob))

        if not ba_sp:
            continue
        if model_name == "xgb":
            ba_xgb_dict[sp] = list(ba_sp)  # H4: store XGB fold BAs for BH correction
        mean_ba_sp, lo_sp, hi_sp = bootstrap_ci(np.array(all_yt_sp), np.array(all_yp_sp))
        results_dict[sp] = {"ba": mean_ba_sp, "lo": lo_sp, "hi": hi_sp,
                            "auroc": np.mean(auc_sp)}

# Print comparison table  -  load RF results from parquet (C1-corrected)
lr_q2 = {"ecloaceae":0.752,"kpneumoniae":0.719,"paeruginosa":0.645,
          "efaecium":0.512,"saureus":0.470,"abaumannii":0.473}
_rf_df = pd.read_parquet(RES / "q2_rf_results_3460.parquet")
rf_q2  = dict(zip(_rf_df["species"], _rf_df["ba"]))

print(f"\nQ2 comparison (balanced accuracy):")
print(f"  {'Species':<22} {'LR':>6}  {'RF':>6}  {'XGB':>6}  {'LGBM':>6}  Winner")
print("  " + "-" * 68)
for sp in sorted(results_q2_xgb.keys()):
    lrv = lr_q2.get(sp, float("nan"))
    rfv = rf_q2.get(sp, float("nan"))
    xv  = results_q2_xgb.get(sp, {}).get("ba", float("nan"))
    lv  = results_q2_lgbm.get(sp, {}).get("ba", float("nan"))
    winner = max(["LR", "RF", "XGB", "LGBM"],
                 key=lambda m: {"LR": lrv, "RF": rfv, "XGB": xv, "LGBM": lv}[m])
    print(f"  {sp:<22} {lrv:.3f}  {rfv:.3f}  {xv:.3f}  {lv:.3f}  {winner}")


  abaumannii: 30 features after H1 filter (from 265)


  ecloaceae: 68 features after H1 filter (from 265)


  efaecium: 23 features after H1 filter (from 265)


  kpneumoniae: 85 features after H1 filter (from 265)


  paeruginosa: 74 features after H1 filter (from 265)


  saureus: 27 features after H1 filter (from 265)



Q2 comparison (balanced accuracy):
  Species                    LR      RF     XGB    LGBM  Winner
  --------------------------------------------------------------------
  abaumannii             0.473  0.489  0.500  0.500  XGB
  ecloaceae              0.752  0.753  0.824  0.681  XGB
  efaecium               0.512  0.489  0.486  0.493  LR
  kpneumoniae            0.719  0.707  0.789  0.756  XGB
  paeruginosa            0.645  0.677  0.568  0.596  RF
  saureus                0.470  0.514  0.508  0.491  RF


## Section 12b  -  H4: BH correction across Q2 species (XGBoost)

**Audit finding H4:** BH correction required across 6 Q2 species tests.
One-sample t-test per species (XGB fold BAs vs BA=0.5 null), then BH correction.
EF, SA, AB may have few usable folds; n_folds reported for transparency.


In [13]:
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests

null_ba  = 0.5
sp_order = sorted(ba_xgb_dict.keys())
p_raw    = []
for sp in sp_order:
    ba_list = ba_xgb_dict[sp]
    if len(ba_list) >= 2:
        _, pval = ttest_1samp(ba_list, popmean=null_ba, alternative="greater")
        p_raw.append(pval)
    else:
        p_raw.append(float("nan"))

valid_idx   = [i for i, p in enumerate(p_raw) if not (p != p)]
sp_valid    = [sp_order[i] for i in valid_idx]
pvals_valid = [p_raw[i]    for i in valid_idx]

reject_bh, pvals_adj, _, _ = multipletests(pvals_valid, alpha=0.05, method="fdr_bh")
q2_pval_map_xgb = dict(zip(sp_valid, pvals_adj))
q2_praw_map_xgb = dict(zip(sp_valid, pvals_valid))
q2_sig_map_xgb  = dict(zip(sp_valid, reject_bh))

print("H4: Q2 XGB null-baseline significance (one-sample t vs BA=0.5, BH-corrected):")
print(f"  {'Species':<22} {'XGB BA':>7}  {'n_folds':>7}  {'p_raw':>8}  {'p_adj_BH':>10}  {'Sig?':>5}")
print("  " + "-" * 72)
for sp in sp_order:
    ba_v = results_q2_xgb.get(sp, {}).get("ba", float("nan"))
    n_f  = len(ba_xgb_dict.get(sp, []))
    p_r  = q2_praw_map_xgb.get(sp, float("nan"))
    p_a  = q2_pval_map_xgb.get(sp, float("nan"))
    sig  = "YES" if q2_sig_map_xgb.get(sp, False) else "ns"
    print(f"  {sp:<22} {ba_v:.3f}   {n_f:>7}  {p_r:.4f}    {p_a:.4f}      {sig}")


H4: Q2 XGB null-baseline significance (one-sample t vs BA=0.5, BH-corrected):
  Species                 XGB BA  n_folds     p_raw    p_adj_BH   Sig?
  ------------------------------------------------------------------------
  abaumannii             0.500         4  nan    nan      ns
  ecloaceae              0.824         5  0.0000    0.0000      YES
  efaecium               0.486         4  0.1181    0.1477      ns
  kpneumoniae            0.789         5  0.0050    0.0126      YES
  paeruginosa            0.568         5  0.0825    0.1374      ns
  saureus                0.508         5  0.6101    0.6101      ns


## Section 12c  -  M4: McNemar tests for Q2 per-species model comparisons

**Audit finding M4:** Q2 cross-model comparisons were reported as point-estimate BA
differences only. With per-species n ~60--100, point estimates alone are uninformative.

**Approach:** Re-run all four models (LR, RF, XGB, LGBM) on the same GroupedStratifiedKFold
splits per species, collecting aligned per-genome predictions. McNemar pairwise test
for each species: LR vs RF, LR vs XGB, LR vs LGBM, RF vs XGB.

Same H1 sparsity filter (>=5% prevalence) applied. XGB/LGBM use early stopping (inner
80/20 split) consistent with the main Q2 run.


In [14]:
from sklearn.linear_model import LogisticRegression as LR_cls
from sklearn.ensemble import RandomForestClassifier as RF_cls

mc_q2 = {}  # sp -> {model_name: pred_array, "y_true": array}

for sp in sorted(fm["species"].unique()):
    sp_mask = fm["species"].to_numpy(dtype=str) == sp
    fm_sp   = fm[sp_mask]
    mask_q2 = fm_sp["arg_burden_tertile"].isin(["low_ARG", "high_ARG"])
    fm_q2   = fm_sp[mask_q2]
    if len(fm_q2) < 20 or fm_q2["arg_burden_tertile"].nunique() < 2:
        continue
    y_sp   = (fm_q2["arg_burden_tertile"] == "high_ARG").astype(int).values

    feat_prev_sp = fm_q2[FEAT_COLS].mean()
    feat_q2_sp   = feat_prev_sp[feat_prev_sp >= 0.05].index.tolist()
    X_sp         = fm_q2[feat_q2_sp].values
    grp_sp       = fm_q2["phylogroup"].to_numpy(dtype=str)
    cv_sp        = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True,
                                        random_state=RANDOM_STATE)

    n_sp = len(y_sp)
    # sentinel -1 for genomes never in a valid test fold
    preds  = {m: np.full(n_sp, -1, dtype=int)
              for m in ["lr", "rf", "xgb", "lgbm"]}
    y_true = np.full(n_sp, -1, dtype=int)

    for tr, te in cv_sp.split(X_sp, y_sp, groups=grp_sp):
        if len(set(y_sp[te])) < 2:
            continue
        y_true[te] = y_sp[te]

        # LR
        lr_m = LR_cls(max_iter=1000, class_weight="balanced",
                      C=0.1, solver="saga", random_state=RANDOM_STATE)
        lr_m.fit(X_sp[tr], y_sp[tr])
        preds["lr"][te] = lr_m.predict(X_sp[te])

        # RF -- extract params from saved model (best_params not in GB scope)
        rf_q2_params = {k: v for k, v in rf_best.get_params().items()
                        if k in ["n_estimators", "max_depth", "min_samples_leaf",
                                 "max_features", "random_state", "n_jobs"]}
        rf_m = RF_cls(**rf_q2_params, class_weight="balanced")
        rf_m.fit(X_sp[tr], y_sp[tr])
        preds["rf"][te] = rf_m.predict(X_sp[te])

        # XGB + LGBM share inner split
        inner_ss = StratifiedShuffleSplit(n_splits=1, test_size=0.2,
                                          random_state=RANDOM_STATE)
        tr_rel, val_rel = next(inner_ss.split(X_sp[tr], y_sp[tr]))
        tr_i, val_i = tr[tr_rel], tr[val_rel]
        sw_i = compute_sample_weight("balanced", y_sp[tr_i])

        xgb_m = XGBClassifier(**best_params_xgb, n_estimators=300,
                               early_stopping_rounds=20, colsample_bytree=0.8,
                               eval_metric="logloss", verbosity=0,
                               random_state=RANDOM_STATE, n_jobs=-1)
        xgb_m.fit(X_sp[tr_i], y_sp[tr_i], sample_weight=sw_i,
                  eval_set=[(X_sp[val_i], y_sp[val_i])], verbose=False)
        preds["xgb"][te] = xgb_m.predict(X_sp[te])

        lgbm_m = LGBMClassifier(
            **{k: v for k, v in lgbm_params.items() if k != "n_estimators"},
            n_estimators=300)
        lgbm_m.fit(X_sp[tr_i], y_sp[tr_i],
                   eval_set=[(X_sp[val_i], y_sp[val_i])],
                   callbacks=[lgb.early_stopping(20, verbose=False),
                               lgb.log_evaluation(-1)])
        preds["lgbm"][te] = lgbm_m.predict(X_sp[te])

    mask = y_true >= 0
    if mask.sum() < 10:
        continue
    yt = y_true[mask]
    mc_q2[sp] = {"y_true": yt}
    for m in ["lr", "rf", "xgb", "lgbm"]:
        mc_q2[sp][m] = preds[m][mask]

    bas = {m: balanced_accuracy_score(yt, mc_q2[sp][m]) for m in ["lr", "rf", "xgb", "lgbm"]}
    print(f"\n{sp} (n={mask.sum()}):")
    print(f"  {'Model':<8}  BA")
    for m in ["lr", "rf", "xgb", "lgbm"]:
        print(f"  {m.upper():<8}  {bas[m]:.3f}")

    pairs = [("LR", "RF"), ("LR", "XGB"), ("LR", "LGBM"), ("RF", "XGB")]
    print(f"  {'Comparison':<16}  {'delta':>6}  {'b':>4}  {'c':>4}  {'p':>7}  Sig?")
    print("  " + "-" * 52)
    for ma, mb in pairs:
        p, b, c, _ = mcnemar_test(yt, mc_q2[sp][ma.lower()], mc_q2[sp][mb.lower()])
        delta = bas[ma.lower()] - bas[mb.lower()]
        sig = "*" if p < 0.05 else "ns"
        print(f"  {ma + ' vs ' + mb:<16}  {delta:+.3f}   {b:>4}  {c:>4}  {p:.4f}  {sig}")



abaumannii (n=89):
  Model     BA
  LR        0.500
  RF        0.489
  XGB       0.500
  LGBM      0.500
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          +0.011      1     0  1.0000  ns
  LR vs XGB         +0.000      0     0  1.0000  ns
  LR vs LGBM        +0.000      0     0  1.0000  ns
  RF vs XGB         -0.011      0     1  1.0000  ns



ecloaceae (n=97):
  Model     BA
  LR        0.721
  RF        0.753
  XGB       0.824
  LGBM      0.681
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          -0.032      5     8  0.5791  ns
  LR vs XGB         -0.103      6    16  0.0550  ns
  LR vs LGBM        +0.040     18    14  0.5959  ns
  RF vs XGB         -0.071      6    13  0.1687  ns



efaecium (n=99):
  Model     BA
  LR        0.593
  RF        0.489
  XGB       0.486
  LGBM      0.493
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          +0.105     14     4  0.0339  *
  LR vs XGB         +0.107     15     4  0.0218  *
  LR vs LGBM        +0.100     14     6  0.1175  ns
  RF vs XGB         +0.002     11    10  1.0000  ns



kpneumoniae (n=86):
  Model     BA
  LR        0.777
  RF        0.707
  XGB       0.789
  LGBM      0.756
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          +0.069      8     2  0.1138  ns
  LR vs XGB         -0.012      5     6  1.0000  ns
  LR vs LGBM        +0.020     12    10  0.8312  ns
  RF vs XGB         -0.082      2     9  0.0704  ns



paeruginosa (n=120):
  Model     BA
  LR        0.638
  RF        0.677
  XGB       0.568
  LGBM      0.596
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          -0.039      7    12  0.3588  ns
  LR vs XGB         +0.070     17     9  0.1698  ns
  LR vs LGBM        +0.042     16    12  0.5708  ns
  RF vs XGB         +0.109     20     7  0.0209  *



saureus (n=106):
  Model     BA
  LR        0.496
  RF        0.514
  XGB       0.508
  LGBM      0.491
  Comparison         delta     b     c        p  Sig?
  ----------------------------------------------------
  LR vs RF          -0.018      2     4  0.6831  ns
  LR vs XGB         -0.011     22    21  1.0000  ns
  LR vs LGBM        +0.005      6     6  0.7728  ns
  RF vs XGB         +0.006     25    22  0.7705  ns


## Section 13  -  Save results

In [15]:
# Q1 summary
q1_gb = pd.DataFrame([
    {"model": "XGBoost", "ba_mean": mean_ba_xgb, "ba_lo": lo_xgb, "ba_hi": hi_xgb,
     "f1_mean": mean_f1_xgb, "cv": "GroupedStratifiedKFold_5",
     "early_stopping": True, "median_best_iter": int(np.median(best_iters)),
     **{f"param_{k}": v for k, v in best_params_xgb.items()}},
    {"model": "LightGBM", "ba_mean": mean_ba_lgbm, "ba_lo": lo_lgbm, "ba_hi": hi_lgbm,
     "f1_mean": mean_f1_lgbm, "cv": "GroupedStratifiedKFold_5",
     "early_stopping": True, "median_best_iter": int(np.median(best_iters_lgbm)),
     "param_num_leaves": 63, **{f"param_{k}": v for k, v in lgbm_params.items()
                                if k in ["learning_rate", "max_depth", "subsample"]}},
])
q1_gb.to_parquet(RES / "q1_gb_results_3460.parquet")
print("Saved: results/q1_gb_results.parquet")

# Q2 summary (including H4 BH-corrected p-values for XGB)
q2_rows = []
for sp, vals in results_q2_xgb.items():
    q2_rows.append({"species": sp, "model": "XGBoost",
                    "p_adj_bh": q2_pval_map_xgb.get(sp, float("nan")),
                    "p_raw":    q2_praw_map_xgb.get(sp, float("nan")),
                    "sig_bh":   q2_sig_map_xgb.get(sp, False),
                    **vals})
for sp, vals in results_q2_lgbm.items():
    q2_rows.append({"species": sp, "model": "LightGBM", **vals})
pd.DataFrame(q2_rows).to_parquet(RES / "q2_gb_results_3460.parquet")
print("Saved: results/q2_gb_results.parquet")

# Per-genome prediction arrays for Phase 10 McNemar (if needed)
np.save(RES / "mc_pred_xgb_q1.npy", mc_pred_xgb)
np.save(RES / "mc_pred_lgbm_q1.npy", mc_pred_lgbm)
np.save(RES / "mc_pred_rf_q1.npy",   mc_pred_rf)
np.save(RES / "mc_true_q1.npy",      mc_true_xgb)   # same genome ordering
print("Saved: McNemar prediction arrays (results/mc_pred_*.npy)")

print("\nAll outputs saved.")


Saved: results/q1_gb_results.parquet
Saved: results/q2_gb_results.parquet
Saved: McNemar prediction arrays (results/mc_pred_*.npy)

All outputs saved.
